# Task 3: Store Cleaned Data in PostgreSQL

## Objective
Design and populate a relational PostgreSQL database to persistently
store the cleaned and processed review data.

## Schema Design
- **banks** table: Stores metadata about each bank
- **reviews** table: Stores all review data with foreign key to banks

## Database: bank_reviews
## Connection: localhost:5432

In [9]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import warnings
warnings.filterwarnings('ignore')

os.chdir(r"C:\Users\pc\fintech-review-analytics")
print("All imports successful")

All imports successful


In [10]:
# Database connection
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="bank_reviews",
    user="postgres",
    password="postgres123"
)
cursor = conn.cursor()
print("Connected to bank_reviews database successfully!")

Connected to bank_reviews database successfully!


## Schema Creation

We create two tables:
1. **banks** — stores bank metadata (bank_id, bank_name, app_name)
2. **reviews** — stores all review data with foreign key to banks

In [11]:
# Drop tables if they exist (for clean setup)
cursor.execute("DROP TABLE IF EXISTS reviews CASCADE;")
cursor.execute("DROP TABLE IF EXISTS banks CASCADE;")

# Create banks table
cursor.execute("""
    CREATE TABLE banks (
        bank_id SERIAL PRIMARY KEY,
        bank_name VARCHAR(100) NOT NULL UNIQUE,
        app_name VARCHAR(200),
        app_id VARCHAR(200),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
""")

# Create reviews table
cursor.execute("""
    CREATE TABLE reviews (
        review_id SERIAL PRIMARY KEY,
        bank_id INTEGER REFERENCES banks(bank_id),
        review_text TEXT,
        rating INTEGER CHECK (rating BETWEEN 1 AND 5),
        review_date DATE,
        sentiment_label VARCHAR(20),
        sentiment_score FLOAT,
        identified_theme VARCHAR(100),
        source VARCHAR(50) DEFAULT 'Google Play',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
""")

conn.commit()
print("Tables created successfully!")
print("  - banks table")
print("  - reviews table")

Tables created successfully!
  - banks table
  - reviews table


In [12]:
# Bank metadata
banks_data = [
    ("Commercial Bank of Ethiopia", "CBE Mobile Banking", "com.combanketh.mobilebanking"),
    ("Bank of Abyssinia", "BOA Mobile Banking", "com.boa.boaMobileBanking"),
    ("Dashen Bank", "Dashen Super App", "com.dashen.dashensuperapp")
]

cursor.execute("DELETE FROM banks;")
for bank_name, app_name, app_id in banks_data:
    cursor.execute("""
        INSERT INTO banks (bank_name, app_name, app_id)
        VALUES (%s, %s, %s)
        ON CONFLICT (bank_name) DO NOTHING;
    """, (bank_name, app_name, app_id))

conn.commit()
print("Banks inserted successfully!")

# Verify
cursor.execute("SELECT * FROM banks;")
rows = cursor.fetchall()
for row in rows:
    print(f"  bank_id={row[0]}: {row[1]} | {row[2]}")

Banks inserted successfully!
  bank_id=1: Commercial Bank of Ethiopia | CBE Mobile Banking
  bank_id=2: Bank of Abyssinia | BOA Mobile Banking
  bank_id=3: Dashen Bank | Dashen Super App


In [13]:
# Load analyzed reviews
df = pd.read_csv("data/raw/bank_reviews_analyzed.csv")
print(f"Loaded {len(df)} reviews")

# Get bank_id mapping
cursor.execute("SELECT bank_id, bank_name FROM banks;")
bank_map = {row[1]: row[0] for row in cursor.fetchall()}
print(f"Bank mapping: {bank_map}")

# Prepare data for insertion
reviews_data = []
for _, row in df.iterrows():
    bank_id = bank_map.get(row["bank"])
    if bank_id is None:
        continue
    
    reviews_data.append((
        bank_id,
        str(row["review"]) if pd.notna(row["review"]) else None,
        int(row["rating"]) if pd.notna(row["rating"]) else None,
        str(row["date"]) if pd.notna(row["date"]) else None,
        str(row["sentiment_label"]) if pd.notna(row["sentiment_label"]) else None,
        float(row["sentiment_score"]) if pd.notna(row["sentiment_score"]) else None,
        str(row["identified_theme"]) if pd.notna(row["identified_theme"]) else None,
        "Google Play"
    ))

# Bulk insert
execute_values(cursor, """
    INSERT INTO reviews 
    (bank_id, review_text, rating, review_date, sentiment_label, 
     sentiment_score, identified_theme, source)
    VALUES %s
""", reviews_data)

conn.commit()
print(f"Inserted {len(reviews_data)} reviews into database")

Loaded 1452 reviews
Bank mapping: {'Commercial Bank of Ethiopia': 1, 'Bank of Abyssinia': 2, 'Dashen Bank': 3}
Inserted 1452 reviews into database


In [14]:
print("=" * 60)
print("DATA INTEGRITY VERIFICATION")
print("=" * 60)

# Count reviews per bank
cursor.execute("""
    SELECT b.bank_name, COUNT(r.review_id) as review_count
    FROM banks b
    LEFT JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name
    ORDER BY review_count DESC;
""")
print("\nReviews per bank:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} reviews")

# Average rating per bank
cursor.execute("""
    SELECT b.bank_name, 
           ROUND(AVG(r.rating)::numeric, 2) as avg_rating,
           ROUND(AVG(r.sentiment_score)::numeric, 3) as avg_sentiment
    FROM banks b
    JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name
    ORDER BY avg_rating DESC;
""")
print("\nAverage rating and sentiment per bank:")
for row in cursor.fetchall():
    print(f"  {row[0]}: Rating={row[1]} | Sentiment={row[2]}")

# Check for nulls in key columns
cursor.execute("""
    SELECT 
        COUNT(*) FILTER (WHERE review_text IS NULL) as null_reviews,
        COUNT(*) FILTER (WHERE rating IS NULL) as null_ratings,
        COUNT(*) FILTER (WHERE sentiment_label IS NULL) as null_sentiment
    FROM reviews;
""")
row = cursor.fetchone()
print(f"\nNull checks:")
print(f"  Null review_text: {row[0]}")
print(f"  Null rating: {row[1]}")
print(f"  Null sentiment_label: {row[2]}")

# Total reviews
cursor.execute("SELECT COUNT(*) FROM reviews;")
total = cursor.fetchone()[0]
print(f"\nTotal reviews in database: {total}")
print("=" * 60)

DATA INTEGRITY VERIFICATION

Reviews per bank:
  Bank of Abyssinia: 499 reviews
  Dashen Bank: 498 reviews
  Commercial Bank of Ethiopia: 455 reviews

Average rating and sentiment per bank:
  Commercial Bank of Ethiopia: Rating=3.94 | Sentiment=0.237
  Dashen Bank: Rating=3.78 | Sentiment=0.264
  Bank of Abyssinia: Rating=3.28 | Sentiment=0.123

Null checks:
  Null review_text: 0
  Null rating: 0
  Null sentiment_label: 0

Total reviews in database: 1452


In [15]:
schema_sql = """
-- bank_reviews Database Schema
-- Kifiya AI Training Program Week 2
-- Author: Rebika Woldeyesus

-- Banks table: stores metadata about each bank
CREATE TABLE IF NOT EXISTS banks (
    bank_id SERIAL PRIMARY KEY,
    bank_name VARCHAR(100) NOT NULL UNIQUE,
    app_name VARCHAR(200),
    app_id VARCHAR(200),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Reviews table: stores all scraped and analyzed review data
CREATE TABLE IF NOT EXISTS reviews (
    review_id SERIAL PRIMARY KEY,
    bank_id INTEGER REFERENCES banks(bank_id),
    review_text TEXT,
    rating INTEGER CHECK (rating BETWEEN 1 AND 5),
    review_date DATE,
    sentiment_label VARCHAR(20),
    sentiment_score FLOAT,
    identified_theme VARCHAR(100),
    source VARCHAR(50) DEFAULT 'Google Play',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Verification queries
-- SELECT b.bank_name, COUNT(r.review_id) FROM banks b LEFT JOIN reviews r ON b.bank_id = r.bank_id GROUP BY b.bank_name;
-- SELECT b.bank_name, AVG(r.rating), AVG(r.sentiment_score) FROM banks b JOIN reviews r ON b.bank_id = r.bank_id GROUP BY b.bank_name;
"""

os.makedirs("scripts", exist_ok=True)
with open("scripts/schema.sql", "w") as f:
    f.write(schema_sql)

print("Schema saved to scripts/schema.sql")

Schema saved to scripts/schema.sql


In [16]:
cursor.close()
conn.close()
print("Database connection closed successfully")
print("\nTask 3 Complete!")
print(f"  Database: bank_reviews")
print(f"  Tables: banks, reviews")
print(f"  Total records: {len(reviews_data)}")

Database connection closed successfully

Task 3 Complete!
  Database: bank_reviews
  Tables: banks, reviews
  Total records: 1452
